   
# Approval Check — Epic on FHIR Model

Deployment job **approval task** (task name must start with 'approval').

Checks Unity Catalog tags for manual approval before promoting the model to
"champion" and updating the serving endpoint.

**Approval pattern (MLflow 3)**: The tag key **must match the job task name**
(`approval_check`). This task checks once and **fails immediately** if not
approved. When an approver clicks "Approve" in the UC model version UI, the
system sets tag `approval_check = Approved` and **auto-repairs** the job run,
resuming from this task.

**Required job parameters**: `model_name`, `model_version`

This task **fails** if the approval tag is missing or not 'approved', which blocks
the downstream deployment task. No polling or timeout — the auto-repair mechanism
handles re-execution.

In [0]:
dbutils.widgets.text("model_name", "", "Model Name (catalog.schema.model)")
dbutils.widgets.text("model_version", "", "Model Version")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")

assert model_name, "model_name parameter is required"
assert model_version, "model_version parameter is required"

print(f"Checking approval for {model_name} v{model_version}")

In [0]:
from mlflow import MlflowClient

# MLflow 3 convention: the tag key MUST match the deployment job's approval
# task_key so the UC model-version UI "Approve" button and auto-repair work.
APPROVAL_TAG_KEY = "approval_check"

client = MlflowClient()
model_version_details = client.get_model_version(model_name, model_version)
tags = model_version_details.tags or {}
approval_tag = tags.get(APPROVAL_TAG_KEY, "")

if approval_tag.lower() == "approved":
    print(f"\u2713 Model version {model_version} approved for deployment")
elif approval_tag.lower() == "rejected":
    raise ValueError(
        f"Model version {model_version} was explicitly rejected.\n"
        f"  Tag: {APPROVAL_TAG_KEY} = {approval_tag}\n"
        f"  To re-approve, update the tag to 'approved' and re-run the deployment job."
    )
else:
    raise RuntimeError(
        f"Model version {model_version} has not been approved yet.\n"
        f"  Tag: {APPROVAL_TAG_KEY} = {approval_tag!r}\n"
        f"To approve:\n"
        f"  1. Unity Catalog UI: Browse to {model_name} v{model_version} > click 'Approve' in the deployment job sidebar\n"
        f"  2. API: MlflowClient().set_model_version_tag('{model_name}', '{model_version}', '{APPROVAL_TAG_KEY}', 'approved')\n"
        f"  3. CLI: databricks api post /api/2.0/mlflow/unity-catalog/model-versions/set-tag "
        f"--json '{{\"name\": \"{model_name}\", \"version\": \"{model_version}\", \"key\": \"{APPROVAL_TAG_KEY}\", \"value\": \"approved\"}}'"
    )

In [0]:
import json

dbutils.notebook.exit(json.dumps({
    "model_name": model_name,
    "model_version": model_version,
    "approval": "passed",
    "approval_tag": approval_tag,
}))